# Deep Agent

`uv add deepagents`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from deepagents import create_deep_agent
from langchain_tavily import TavilySearch


SYSTEM_PROMPT = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.
Answer in KOREAN

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

# 모델이 자체적으로 인터넷 검색을 지원 할 경우에는, 아래처럼 tool로 작성 가능
# OpenAI 빌트인 검색
internet_search = {'type': 'web_search'}

tavily = TavilySearch()

agent = create_deep_agent(
    model='openai:gpt-5.4-mini',
    tools=[tavily],
    system_prompt=SYSTEM_PROMPT
)

agent

In [ ]:
result = agent.invoke({
    'messages': [
        {'role': 'user', 'content': 'Langchain, Langgraph, DeepAgent 가 뭔지 조사 하고 설명해줘'}
    ]
})

## Deep Research Agent
### Agent
- Prompt
    - Main Agent 프롬프트
    - Sub Agent 프롬프트
    - Subagent 위임 지침
- Tools
    - Tavily Search
    - Think
- Subagent

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### Tools
```
uv add markdownify tavily
```

In [ ]:
# Tool
"""Research Tools.

This module provides search and content processing utilities for the research agent,
using Tavily for URL discovery and fetching full webpage content.
"""

import httpx
from langchain_core.tools import InjectedToolArg, tool
from markdownify import markdownify
from tavily import TavilyClient
from typing_extensions import Annotated, Literal

tavily_client = TavilyClient()


# 단순 함수 (아래 tool 에서 사용)
def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
    """Fetch and convert webpage content to markdown.

    Args:
        url: URL to fetch
        timeout: Request timeout in seconds

    Returns:
        Webpage content as markdown
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        response = httpx.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return markdownify(response.text)
    except Exception as e:
        return f"Error fetching content from {url}: {str(e)}"


@tool(parse_docstring=True)
def tavily_search(
    query: str,
    max_results: Annotated[int, InjectedToolArg] = 1,
    topic: Annotated[
        Literal["general", "news", "finance"], InjectedToolArg
    ] = "general",
) -> str:
    """Search the web for information on a given query.

    Uses Tavily to discover relevant URLs, then fetches and returns full webpage content as markdown.

    Args:
        query: Search query to execute
        max_results: Maximum number of results to return (default: 1)
        topic: Topic filter - 'general', 'news', or 'finance' (default: 'general')

    Returns:
        Formatted search results with full webpage content
    """
    # Use Tavily to discover URLs
    search_results = tavily_client.search(
        query,
        max_results=max_results,
        topic=topic,
    )

    # Fetch full content for each URL
    result_texts = []
    for result in search_results.get("results", []):
        url = result["url"]
        title = result["title"]

        # Fetch webpage content
        content = fetch_webpage_content(url)

        result_text = f"""## {title}
**URL:** {url}

{content}

---
"""
        result_texts.append(result_text)

    # Format final response
    response = f"""🔍 Found {len(result_texts)} result(s) for '{query}':

{chr(10).join(result_texts)}"""

    return response


@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """Tool for strategic reflection on research progress and decision-making.

    Use this tool after each search to analyze results and plan next steps systematically.
    This creates a deliberate pause in the research workflow for quality decision-making.

    When to use:
    - After receiving search results: What key information did I find?
    - Before deciding next steps: Do I have enough to answer comprehensively?
    - When assessing research gaps: What specific information am I still missing?
    - Before concluding research: Can I provide a complete answer now?

    Reflection should address:
    1. Analysis of current findings - What concrete information have I gathered?
    2. Gap assessment - What crucial information is still missing?
    3. Quality evaluation - Do I have sufficient evidence/examples for a good answer?
    4. Strategic decision - Should I continue searching or provide my answer?

    Args:
        reflection: Your detailed reflection on research progress, findings, gaps, and next steps

    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    return f"Reflection recorded: {reflection}"

In [ ]:
"""Prompt templates and tool descriptions for the research deepagent."""

RESEARCH_WORKFLOW_INSTRUCTIONS = """# Research Workflow

Follow this workflow for all research requests:

1. **Plan**: Create a todo list with write_todos to break down the research into focused tasks
2. **Save the request**: Use write_file() to save the user's research question to `/research_request.md`
3. **Research**: Delegate research tasks to sub-agents using the task() tool - ALWAYS use sub-agents for research, never conduct research yourself
4. **Synthesize**: Review all sub-agent findings and consolidate citations (each unique URL gets one number across all findings)
5. **Write Report**: Write a comprehensive final report to `/final_report.md` (see Report Writing Guidelines below)
6. **Verify**: Read `/research_request.md` and confirm you've addressed all aspects with proper citations and structure

## Research Planning Guidelines
- Batch similar research tasks into a single TODO to minimize overhead
- For simple fact-finding questions, use 1 sub-agent
- For comparisons or multi-faceted topics, delegate to multiple parallel sub-agents
- Each sub-agent should research one specific aspect and return findings

## Report Writing Guidelines

When writing the final report to `/final_report.md`, follow these structure patterns:

**For comparisons:**
1. Introduction
2. Overview of topic A
3. Overview of topic B
4. Detailed comparison
5. Conclusion

**For lists/rankings:**
Simply list items with details - no introduction needed:
1. Item 1 with explanation
2. Item 2 with explanation
3. Item 3 with explanation

**For summaries/overviews:**
1. Overview of topic
2. Key concept 1
3. Key concept 2
4. Key concept 3
5. Conclusion

**General guidelines:**
- Use clear section headings (## for sections, ### for subsections)
- Write in paragraph form by default - be text-heavy, not just bullet points
- Do NOT use self-referential language ("I found...", "I researched...")
- Write as a professional report without meta-commentary
- Each section should be comprehensive and detailed
- Use bullet points only when listing is more appropriate than prose

**Citation format:**
- Cite sources inline using [1], [2], [3] format
- Assign each unique URL a single citation number across ALL sub-agent findings
- End report with ### Sources section listing each numbered source
- Number sources sequentially without gaps (1,2,3,4...)
- Format: [1] Source Title: URL (each on separate line for proper list rendering)
- Example:

  Some important finding [1]. Another key insight [2].

  ### Sources
  [1] AI Research Paper: https://example.com/paper
  [2] Industry Analysis: https://example.com/analysis
"""

RESEARCHER_INSTRUCTIONS = """You are a research assistant conducting research on the user's input topic. For context, today's date is {date}.

<Task>
Your job is to use tools to gather information about the user's input topic.
You can use any of the research tools provided to you to find resources that can help answer the research question. 
You can call these tools in series or in parallel, your research is conducted in a tool-calling loop.
</Task>

<Available Research Tools>
You have access to two specific research tools:
1. **tavily_search**: For conducting web searches to gather information
2. **think_tool**: For reflection and strategic planning during research
**CRITICAL: Use think_tool after each search to reflect on results and plan next steps**
</Available Research Tools>

<Instructions>
Think like a human researcher with limited time. Follow these steps:

1. **Read the question carefully** - What specific information does the user need?
2. **Start with broader searches** - Use broad, comprehensive queries first
3. **After each search, pause and assess** - Do I have enough to answer? What's still missing?
4. **Execute narrower searches as you gather information** - Fill in the gaps
5. **Stop when you can answer confidently** - Don't keep searching for perfection
</Instructions>

<Hard Limits>
**Tool Call Budgets** (Prevent excessive searching):
- **Simple queries**: Use 2-3 search tool calls maximum
- **Complex queries**: Use up to 5 search tool calls maximum
- **Always stop**: After 5 search tool calls if you cannot find the right sources

**Stop Immediately When**:
- You can answer the user's question comprehensively
- You have 3+ relevant examples/sources for the question
- Your last 2 searches returned similar information
</Hard Limits>

<Show Your Thinking>
After each search tool call, use think_tool to analyze the results:
- What key information did I find?
- What's missing?
- Do I have enough to answer the question comprehensively?
- Should I search more or provide my answer?
</Show Your Thinking>

<Final Response Format>
When providing your findings back to the orchestrator:

1. **Structure your response**: Organize findings with clear headings and detailed explanations
2. **Cite sources inline**: Use [1], [2], [3] format when referencing information from your searches
3. **Include Sources section**: End with ### Sources listing each numbered source with title and URL

Example:
```
## Key Findings

Context engineering is a critical technique for AI agents [1]. Studies show that proper context management can improve performance by 40% [2].

### Sources
[1] Context Engineering Guide: https://example.com/context-guide
[2] AI Performance Study: https://example.com/study
```

The orchestrator will consolidate citations from all sub-agents into the final report.
</Final Response Format>
"""

TASK_DESCRIPTION_PREFIX = """Delegate a task to a specialized sub-agent with isolated context. Available agents for delegation are:
{other_agents}
"""

SUBAGENT_DELEGATION_INSTRUCTIONS = """# Sub-Agent Research Coordination

Your role is to coordinate research by delegating tasks from your TODO list to specialized research sub-agents.

## Delegation Strategy

**DEFAULT: Start with 1 sub-agent** for most queries:
- "What is quantum computing?" → 1 sub-agent (general overview)
- "List the top 10 coffee shops in San Francisco" → 1 sub-agent
- "Summarize the history of the internet" → 1 sub-agent
- "Research context engineering for AI agents" → 1 sub-agent (covers all aspects)

**ONLY parallelize when the query EXPLICITLY requires comparison or has clearly independent aspects:**

**Explicit comparisons** → 1 sub-agent per element:
- "Compare OpenAI vs Anthropic vs DeepMind AI safety approaches" → 3 parallel sub-agents
- "Compare Python vs JavaScript for web development" → 2 parallel sub-agents

**Clearly separated aspects** → 1 sub-agent per aspect (use sparingly):
- "Research renewable energy adoption in Europe, Asia, and North America" → 3 parallel sub-agents (geographic separation)
- Only use this pattern when aspects cannot be covered efficiently by a single comprehensive search

## Key Principles
- **Bias towards single sub-agent**: One comprehensive research task is more token-efficient than multiple narrow ones
- **Avoid premature decomposition**: Don't break "research X" into "research X overview", "research X techniques", "research X applications" - just use 1 sub-agent for all of X
- **Parallelize only for clear comparisons**: Use multiple sub-agents when comparing distinct entities or geographically separated data

## Parallel Execution Limits
- Use at most {max_concurrent_research_units} parallel sub-agents per iteration
- Make multiple task() calls in a single response to enable parallel execution
- Each sub-agent returns findings independently

## Research Limits
- Stop after {max_researcher_iterations} delegation rounds if you haven't found adequate sources
- Stop when you have sufficient information to answer comprehensively
- Bias towards focused research over exhaustive exploration"""

In [ ]:
"""Research Agent - Standalone script for LangGraph deployment.

This module creates a deep research agent with custom tools and prompts
for conducting web research with strategic thinking and context management.
"""

from datetime import datetime

from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent

# Limits
max_concurrent_research_units = 3
max_researcher_iterations = 3

# Get current date
current_date = datetime.now().strftime("%Y-%m-%d")

# Combine orchestrator instructions (RESEARCHER_INSTRUCTIONS only for sub-agents)
INSTRUCTIONS = (
    RESEARCH_WORKFLOW_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
        max_concurrent_research_units=max_concurrent_research_units,
        max_researcher_iterations=max_researcher_iterations,
    )
)

# 서브 에이전트
sub_model = init_chat_model('openai:gpt-4.1-mini', temperature=0)
# Create research sub-agent
research_sub_agent = {
    "name": "research-agent",
    # 메인 에이전트(오케스트레이터)에게 알려주는 내용
    "description": "Delegate research to the sub-agent researcher. Only give this researcher one topic at a time.",
    "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
    "tools": [tavily_search, think_tool],
    "model": sub_model,
}


# 메인 에이전트
model = init_chat_model(model="openai:gpt-5.4-mini", temperature=0.0)

In [ ]:
from deepagents.backends import FilesystemBackend

# Create the agent
deep_research_agent = create_deep_agent(
    model=model,
    tools=[tavily_search, think_tool],
    system_prompt=INSTRUCTIONS,
    subagents=[research_sub_agent],
    # 실제 파일/폴더 접근 권한은 위험할 수 있다. 주의!
    backend=FilesystemBackend(
        root_dir='./deepagent-files/'
    )
)

In [ ]:
# 메인 에이전트용 프롬프트
print(INSTRUCTIONS)
# 서브 에이전트용 프롬프트
print(RESEARCHER_INSTRUCTIONS)

In [ ]:
from langchain.messages import HumanMessage

result = deep_research_agent.invoke({
    'messages': [
        HumanMessage('AI 에이전트와 컨텍스트 엔지니어링에 대해 조사해줘')
    ]
})

In [ ]:
result['messages'][-1].pretty_print()

## Data Analysis Agent
```
uv add slack-sdk langchain-daytona
```
- 데이터 분석 에이전트 -> 저번달 매출 평균
- 머신 러닝 에이전트 -> 지금까지 기반으로 다음달 매출 예측
- 둘 다 파이썬 스크립트(코드) 작성 후 실행.
- 이때 실행은 위험함 -> 나쁜 요청이 들어와서 파일 삭제하는 스크립트 실행 가능성 + 패키지 설치도 영향 x
- 안전한 격리 환경을 만들고, 해당 환경에서 코드를 실행(sandbox)

In [23]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
import os
from daytona import Daytona
from langchain_daytona import DaytonaSandbox

sandbox = Daytona().create()
backend = DaytonaSandbox(sandbox=sandbox)

local_csv_path = "./data.csv" 
if os.path.exists(local_csv_path):
    with open(local_csv_path, "rb") as f:
        csv_bytes = f.read()

    # 업로드 할 경로
    target_agent_path = "/home/daytona/dataset.csv"
    results = backend.upload_files([
        (target_agent_path, csv_bytes)
    ])
    if results[0].error:
        print(f"❌ 에러: {results[0].error}")
    else:
        print(f"✅ 내 컴퓨터의 '{local_csv_path}' 백엔드 파일 경로:'{target_agent_path}'로 업로드 성공했습니다!")
else:
    print(f"❌ 에러: '{local_csv_path}' 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요.")

✅ 내 컴퓨터의 './data.csv' 백엔드 파일 경로:'/home/daytona/dataset.csv'로 업로드 성공했습니다!


In [ ]:
from langchain_core.utils.uuid import uuid7

from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model="openai:gpt-5.4-mini",
    tools=[],
    backend=backend,
    checkpointer=checkpointer,
    # 모든 model 활동마다, Todo 를 확인하도록 강제하는 MiddleWare를 추가
    middleware=[TodoListMiddleware()],
)

thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

In [26]:
input_message = {
    "role": "user",
    "content": (
        "Analyze /home/daytona/dataset.csv in the current dir and generate a beautiful plot. "
        "When finished, send your analysis and the plot to Slack using the tool."
    ),
}
stream = agent.stream_events(
    {"messages": [input_message]},
    config,
    version="v3",
)
for snapshot in stream.values:
    snapshot["messages"][-1].pretty_print()

================================ Human Message =================================

Analyze /home/daytona/dataset.csv in the current dir and generate a beautiful plot. When finished, send your analysis and the plot to Slack using the tool.
================================== Ai Message ==================================

[{'type': 'tool_call', 'id': 'call_NwwNdP12yLywIUdlZRQcPf7V', 'name': 'write_todos', 'args': {'todos': [{'content': 'Inspect dataset.csv structure and contents', 'status': 'in_progress'}, {'content': 'Analyze data and determine an appropriate plot', 'status': 'pending'}, {'content': 'Generate the plot file', 'status': 'pending'}, {'content': 'Send analysis and plot to Slack using the available tool', 'status': 'pending'}]}, 'extras': {'item_id': 'fc_087166fad7406bf3006aa7a1ab5a6887d0be4387ba6f37d551'}}]
Tool Calls:
  write_todos (call_NwwNdP12yLywIUdlZRQcPf7V)
 Call ID: call_NwwNdP12yLywIUdlZRQcPf7V
  Args:
    todos: [{'content': 'Inspect dataset.csv structure and conten